In [ ]:
# imports

from src.config import settings
from src.api import BOEDownloader
from src.preprocess import Preprocesador, generar_esquemas
from src.semantic_schemas import (
    NormaSchema,
    UserQuerySchema,
    schema_to_anthropic,
    anthropic_to_md,
    _norma_include,
)
from src.llm import Llm

import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv

load_dotenv()

print("✓ Imports OK")
print(f"  Neo4j URI:  {settings.neo4j.uri}")
print(f"  LLM model:  {settings.llm.model}")
print(f"  Max exchanges: {settings.llm.max_exchanges}")

✓ Imports OK
  Neo4j URI:  bolt://localhost:7687
  LLM model:  claude-haiku-4-5
  Max exchanges: 2


In [ ]:
# API

api = BOEDownloader()
resumen = api.descargar_masivo()
print(json.dumps(resumen, indent=2, ensure_ascii=False))

In [ ]:
# Preprocesar — parsear XMLs y cargar en Neo4j

preprocesador = Preprocesador()
resumen = preprocesador.preprocesar_todo()
print(json.dumps(resumen, indent=2, ensure_ascii=False))

In [2]:
def borrar_nodos_dinamicos_bbdd() -> dict:
    """Elimina todos los nodos :UserQuery y aristas :RESULT_EDGE del grafo.

    Útil para limpiar el grafo dinámico entre sesiones de prueba.

    Returns:
        Resumen con el número de nodos y aristas eliminados.
    """
    from neo4j import GraphDatabase

    driver = GraphDatabase.driver(
        settings.neo4j.uri,
        auth=(settings.neo4j.user, settings.neo4j.password),
    )
    with driver.session(database=settings.neo4j.database) as session:
        result = session.run("MATCH (q:UserQuery) DETACH DELETE q RETURN count(q) AS eliminados")
        eliminados = result.single()["eliminados"]
    driver.close()
    return {"nodos_eliminados": eliminados}


print(borrar_nodos_dinamicos_bbdd())

{'nodos_eliminados': 0}


In [ ]:
# Neo4j — stats rápidas del grafo
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    settings.neo4j.uri,
    auth=(settings.neo4j.user, settings.neo4j.password),
)
with driver.session(database=settings.neo4j.database) as session:
    stats = {
        "total_normas": session.run("MATCH (n:Norma) RETURN count(n) AS c").single()["c"],
        "vigentes": session.run("MATCH (n:Norma {vigente:true}) RETURN count(n) AS c").single()[
            "c"
        ],
        "derogan": session.run("MATCH ()-[:DEROGA]->() RETURN count(*) AS c").single()["c"],
        "modifican": session.run("MATCH ()-[:MODIFICA]->() RETURN count(*) AS c").single()["c"],
        "citan": session.run("MATCH ()-[:CITA]->() RETURN count(*) AS c").single()["c"],
        "user_queries": session.run("MATCH (q:UserQuery) RETURN count(q) AS c").single()["c"],
    }
driver.close()

print("Stats del grafo:")
for k, v in stats.items():
    print(f"  {k}: {v:,}")

In [2]:
# Generar esquemas semánticos

generar_esquemas(settings.preprocess.ontology_dir)

2026-06-01 14:43:05 [info     ] 
Esquemas creados              relaciones=3 semantic_dir=/home/jorgee/reversa/ontology/semantic-layer


In [3]:
# LLM — prueba de una sola pregunta
async def test_llm_simple():
    llm = Llm()
    pregunta = "¿Cuántas normas vigentes hay en el grafo?"
    print(f"Pregunta: {pregunta}\n")
    print("Respuesta: ", end="", flush=True)
    async for token in llm.responder(pregunta):
        print(token, end="", flush=True)
    print()
    llm.close()


await test_llm_simple()

Pregunta: ¿Cuántas normas vigentes hay en el grafo?

Respuesta: El grafo contiene **9.908 normas vigentes** actualmente.


In [ ]:
# LLM — prueba de conversación multi-turno (max_exchanges)
async def test_llm_conversacion():
    llm = Llm()
    preguntas = [
        "¿Cuántas leyes vigentes hay en el grafo?",
        "¿Y cuántos reales decretos?",
        "¿Cuál es la norma más citada?",  # este turno descarta el primero del historial
    ]
    for i, pregunta in enumerate(preguntas, 1):
        print(f"--- Turno {i}: {pregunta}")
        print("Respuesta: ", end="", flush=True)
        async for token in llm.responder(pregunta):
            print(token, end="", flush=True)
        print(f"\n[historial: {len(llm._history)} exchange(s)]\n")
    llm.close()


await test_llm_conversacion()

In [5]:
# Esquemas — inspección y regeneración
norma_schema = schema_to_anthropic(NormaSchema, include=_norma_include(settings.parse))

print("=== Nodo :Norma (Markdown) ===\n")
print(anthropic_to_md(norma_schema))

print("=== Nodo :UserQuery (Markdown) ===\n")
print(anthropic_to_md(schema_to_anthropic(UserQuerySchema)))

print("=== Norma JSON (Anthropic) ===")
print(json.dumps(norma_schema, indent=2, ensure_ascii=False))

=== Nodo :Norma (Markdown) ===

# :Norma

Nodo principal del grafo. Una norma consolidada del boletín oficial.

| Atributo | Tipo | Obligatorio | Descripción | Ejemplo |
|---|---|---|---|---|
| id | string | sí | Identificador del boletín oficial (e.g. BOE-A-2015-10565) | `BOE-A-2015-10565` |
| fecha_actualizacion | string | no | Fecha de última actualización ISO-8601 | `20251201T120000Z` |
| ambito_codigo | int | no | Código del ámbito territorial (1=Estatal, 2=Autonómico…) | `1` |
| ambito | string | no | Texto del ámbito territorial (e.g. Estatal) | `Estatal` |
| departamento_codigo | int | no | Código del departamento emisor | `3681` |
| departamento | string | no | Nombre del departamento emisor | `Jefatura del Estado` |
| rango_codigo | int | no | Código del rango normativo | `1300` |
| rango | string | no | Texto del rango (e.g. Ley, Real Decreto) | `Ley` |
| fecha_disposicion | string | no | Fecha de disposición YYYY-MM-DD | `2015-10-01` |
| numero_oficial | string | no | Númer